In [1]:
import requests
import os
from pathlib import Path

data_dir = Path("data/raw")
data_dir.mkdir(parents=True, exist_ok=True)

taxi_file = data_dir / "taxi_data.parquet"
zone_file = data_dir / "taxi_zone_lookup.csv"

# Only download taxi data if not already present
if not taxi_file.exists():
    url1 = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
    response1 = requests.get(url1)
    response1.raise_for_status()
    taxi_file.write_bytes(response1.content)
    print("Taxi data downloaded.")
else:
    print("Taxi data already exists. Skipping download.")

# Only download zone lookup if not already present
if not zone_file.exists():
    url2 = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
    response2 = requests.get(url2)
    response2.raise_for_status()
    zone_file.write_bytes(response2.content)
    print("Zone lookup downloaded.")
else:
    print("Zone lookup already exists. Skipping download.")

print("Download check complete!")

Taxi data already exists. Skipping download.
Zone lookup already exists. Skipping download.
Download check complete!


## Part 1: Distributed Data Processing with Spark

Task 1.1

In [2]:
import os
import subprocess

os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

print(subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT).decode())
print("JAVA_HOME =", os.environ.get("JAVA_HOME"))

java version "17.0.12" 2024-07-16 LTS
Java(TM) SE Runtime Environment (build 17.0.12+8-LTS-286)
Java HotSpot(TM) 64-Bit Server VM (build 17.0.12+8-LTS-286, mixed mode, sharing)

JAVA_HOME = C:\Program Files\Java\jdk-17


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TaxiAnalysis") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

df = spark.read.parquet("data/raw/taxi_data.parquet")
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [4]:
row_count = df.count()
partition_count = df.rdd.getNumPartitions()

print("Rows:", row_count)
print("Partitions:", partition_count)

Rows: 2964624
Partitions: 12


In [5]:
import time
import pandas as pd

# Pandas load time
start = time.time()
df = pd.read_parquet("data/raw/taxi_data.parquet")
pandas_time = time.time() - start

# Spark load time
start = time.time()
df = spark.read.parquet("data/raw/taxi_data.parquet")
df.count()
spark_time = time.time() - start

print(f"Spark load time: {spark_time:.2f} seconds")
print(f"Pandas load time: {pandas_time:.2f} seconds")

Spark load time: 0.19 seconds
Pandas load time: 0.12 seconds


task 1.2

In [ ]:
from pyspark.sql import functions as F

initial_count = df.count()
print(f"Initial rows: {initial_count}")

# setting critical columns
critical_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "trip_distance",
    "tip_amount"
]

# drop nulls
df_clean = df.dropna(subset=critical_cols)
after_nulls = df_clean.count()
print(f"Rows after null removal: {after_nulls}")
print(f"Rows removed for nulls: {initial_count - after_nulls}")


# clean rows
df_clean = df_clean.filter(F.col("trip_distance") > 0)
after_distance = df_clean.count()
print(f"Rows removed for non-positive distance: {after_nulls - after_distance}")

df_clean = df_clean.filter(F.col("fare_amount") >= 0)
after_negative_fare = df_clean.count()
print(f"Rows removed for negative fare: {after_distance - after_negative_fare}")

df_clean = df_clean.filter(F.col("fare_amount") <= 500)
after_high_fare = df_clean.count()
print(f"Rows removed for fare > 500: {after_negative_fare - after_high_fare}")

df_clean = df_clean.filter(
    F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))
after_bad_time = df_clean.count()
print(f"Rows removed for dropoff before pickup: {after_high_fare - after_bad_time}")

Initial rows: 2964624
Rows after null removal: 2964624
Rows removed for nulls: 0
Rows removed for non-positive distance: 60371
Rows removed for negative fare: 34065
Rows removed for fare > 500: 30
Rows removed for dropoff before pickup: 56


Derived Columns

In [9]:
from pyspark.sql import functions as F

df_clean = df_clean.withColumn(
    "trip_duration_minutes",
    (
        F.unix_timestamp("tpep_dropoff_datetime") -
        F.unix_timestamp("tpep_pickup_datetime")
    ) / 60.0)

df_clean = df_clean.withColumn(
    "trip_speed_mph",
    F.when(
        F.col("trip_duration_minutes") > 0,
        F.col("trip_distance") / (F.col("trip_duration_minutes") / 60.0)
    ).otherwise(0.0))

df_clean = df_clean.withColumn(
    "pickup_hour",
    F.hour("tpep_pickup_datetime"))

df_clean = df_clean.withColumn(
    "pickup_day_of_week",
    F.dayofweek("tpep_pickup_datetime"))

df_clean = df_clean.withColumn(
    "tip_percentage",
    F.when(
        F.col("fare_amount") > 1, 
        (F.col("tip_amount") / F.col("fare_amount")) * 100.0
    ).otherwise(0.0))

final_count = df_clean.count()
print(f"Final cleaned rows: {final_count}")

df_clean.select(
    "trip_duration_minutes",
    "trip_speed_mph",
    "pickup_hour",
    "pickup_day_of_week",
    "tip_percentage"
).show(5, truncate=False)

Final cleaned rows: 2870102
+---------------------+------------------+-----------+------------------+------------------+
|trip_duration_minutes|trip_speed_mph    |pickup_hour|pickup_day_of_week|tip_percentage    |
+---------------------+------------------+-----------+------------------+------------------+
|19.8                 |5.212121212121212 |0          |2                 |0.0               |
|6.6                  |16.363636363636363|0          |2                 |37.5              |
|17.916666666666668   |15.739534883720932|0          |2                 |12.875536480686694|
|8.3                  |10.120481927710843|0          |2                 |20.0              |
|6.1                  |7.868852459016395 |0          |2                 |40.50632911392405 |
+---------------------+------------------+-----------+------------------+------------------+
only showing top 5 rows


In [10]:
df_clean.select(
    F.min("trip_duration_minutes"),
    F.min("trip_speed_mph"),
    F.min("tip_percentage"),
    F.max("tip_percentage")
).show()

+--------------------------+-------------------+-------------------+-------------------+
|min(trip_duration_minutes)|min(trip_speed_mph)|min(tip_percentage)|max(tip_percentage)|
+--------------------------+-------------------+-------------------+-------------------+
|                       0.0|                0.0|                0.0|             2500.0|
+--------------------------+-------------------+-------------------+-------------------+



task 1.3

In [12]:
df_clean.createOrReplaceTempView("taxi_trips")

Query 1: What are the top 10 busiest pickup hours, and what is the average fare and tip 
percentage for each? 

In [ ]:
query1 = """
SELECT
    pickup_hour,
    COUNT(*) AS trip_count,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
FROM taxi_trips
GROUP BY pickup_hour
ORDER BY trip_count DESC
LIMIT 10
"""

busiest_locations = spark.sql(query1)
busiest_locations.show(truncate=False)

+-----------+----------+--------+------------------+
|pickup_hour|trip_count|avg_fare|avg_tip_percentage|
+-----------+----------+--------+------------------+
|18         |206284    |17.01   |22.77             |
|17         |200315    |18.12   |22.33             |
|16         |184971    |19.46   |21.83             |
|15         |184009    |19.11   |19.8              |
|19         |178812    |17.63   |22.8              |
|14         |178031    |19.27   |19.78             |
|13         |165361    |18.42   |19.78             |
|12         |159916    |17.8    |19.74             |
|21         |155915    |18.29   |21.88             |
|20         |155561    |18.05   |22.1              |
+-----------+----------+--------+------------------+



Query 2: Which day of the week has the highest average trip speed? Include average distance and duration. 

In [ ]:
query2 = """
SELECT
    pickup_day_of_week,
    ROUND(AVG(trip_speed_mph), 2) AS avg_speed_mph,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(trip_duration_minutes), 2) AS avg_duration_minutes
FROM taxi_trips
GROUP BY pickup_day_of_week
ORDER BY avg_speed_mph DESC
"""

highest_avg_trip_speed = spark.sql(query2)
highest_avg_trip_speed.show(truncate=False)

Query 3: Using a window function, rank the top 5 pickup locations by total revenue for each day of the week.

In [ ]:
query3 = """
WITH revenue_by_day AS (
    SELECT
        pickup_day_of_week,
        PULocationID,
        ROUND(SUM(total_amount), 2) AS total_revenue
    FROM taxi_trips
    GROUP BY pickup_day_of_week, PULocationID
),
ranked_locations AS (
    SELECT
        pickup_day_of_week,
        PULocationID,
        total_revenue,
        ROW_NUMBER() OVER (
            PARTITION BY pickup_day_of_week
            ORDER BY total_revenue DESC
        ) AS revenue_rank
    FROM revenue_by_day
)
SELECT
    pickup_day_of_week,
    PULocationID,
    total_revenue,
    revenue_rank
FROM ranked_locations
WHERE revenue_rank <= 5
ORDER BY pickup_day_of_week, revenue_rank
"""

q3_result = spark.sql(query3)
q3_result.show(50, truncate=False)

Query 4: Calculate the cumulative trip count by hour of day (running total from hour 0 to 23). At what hour does the cumulative count surpass 50% of daily trips? 

Query 5: Compare average fare, distance, and tip percentage between short trips (<2 miles), medium trips (2–10 miles), and long trips (>10 miles). Which category has the highest tip percentage? 